In [1]:
#IMPORTS
import pandas as pd
from datetime import datetime
import time
from time import sleep

from agents import *
from host_demo import thermal_classification
from papers import *
import os
from datetime import datetime
import json

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# get accession
df=pd.read_csv('data/example/example_data.csv')
accessions=df['Accession']

In [48]:
#Create library


# Create csv file
with open('results/library/library_summary.csv', 'w') as csvfile1:
    csvfile1.write(f'accession,number_of_papers,pmc_papers,pm_papers,duration(s)\n')

# Create txt file
with open('results/library/library_curation.txt', 'w') as infile:
    infile.write(f'Summary of the library Curation for example data\n{datetime.now()}\n\n')

# create library for all accessions
full_library_start=time.time()
for accession in accessions:
    # get start time
    start=time.time()
    # Create library
    CreateLibrary(accession)
    #End time
    end=time.time()
    #Duration
    duration=end-start

    #Number of papers
    papers=sorted(os.listdir(f'data/library/{accession}'))

    # Number of PMC papers
    pmc_papers=0
    for paper in papers:
        if paper.startswith('PMC'):
            pmc_papers+=1

    pm_papers=(len(papers)-pmc_papers)-1 # -1 for json

    with open(f'results/library/library_curation.txt','a') as f:
        f.write(f'\n\n\n================{accession}===============\n')
        f.write(f'Number of papers: {len(papers)}\n')
        f.write(f'Number of PMCs: {pmc_papers}\n')
        f.write(f'Number of PM papers: {pm_papers}\n')
        f.write(f'Duration: {round(duration,2)} seconds\n')

        f.write(f'======Paper Summary======\n')
        for paper in papers:
            with open(f'data/library/{accession}/{paper}','r') as infile:
                length=len(infile.read())
            f.write(f'{paper}\nNumber of characters: {length}\n')

    with open(f'results/library/library_summary.csv', 'a') as csvfile:
        csvfile.write(f'{accession},{len(papers)},{pmc_papers},{pm_papers},{duration}\n')




full_library_end=time.time()
full_duration=(full_library_end-full_library_start)/60
with open('results/library/library_curation.txt','a') as f:
    f.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}\n')

Tequatrovirus T4
No Papers associated with ADI87650
Opting for query search
download: s3://pmc-oa-opendata/PMC13048485.1/PMC13048485.1.txt to data/library/ADI87650/PMC13048485.1.txt
download: s3://pmc-oa-opendata/PMC13030558.1/PMC13030558.1.txt to data/library/ADI87650/PMC13030558.1.txt
download: s3://pmc-oa-opendata/PMC13055312.1/PMC13055312.1.txt to data/library/ADI87650/PMC13055312.1.txt
download: s3://pmc-oa-opendata/PMC12929629.1/PMC12929629.1.txt to data/library/ADI87650/PMC12929629.1.txt
download: s3://pmc-oa-opendata/PMC12919812.1/PMC12919812.1.txt to data/library/ADI87650/PMC12919812.1.txt
download: s3://pmc-oa-opendata/PMC12893549.1/PMC12893549.1.txt to data/library/ADI87650/PMC12893549.1.txt
No txts or PDFs found for this paper: PMC12874024
CompletedProcess(args='/usr/local/bin/aws --no-sign-request s3 ls s3://pmc-oa-opendata/PMC12874024.1/', returncode=1, stdout='', stderr='')
download: s3://pmc-oa-opendata/PMC12929587.1/PMC12929587.1.txt to data/library/ADI87650/PMC1292958

In [12]:
# Identify Bacterial Host from metadata

with open('results/host/host_classification.csv', 'w') as csvfile1:
    csvfile1.write(f'accession,host,taxonomic_level,reasoning,duration(m)\n')

with open('results/host/host_classification.txt','w') as f:
    f.write(f'Host classification of example data using metadata\n{datetime.now()}\n\n')

full_Start=time.time()
for accession in accessions:
    print(f'=============={accession}=============')
    with open(f"data/library/{accession}/{accession}.json", "r") as inj:
        metadata = json.load(inj)
        print(f'Metadata: {metadata}')

        # Define the start
    state = {
        'paper_text': None,
        'source_path': None,
        'model': 'qwen3.5',
        'metadata': metadata, # only including metadata
        'host_species': None,
        'reasoning': None,
        'confidence': None,
        'thermal_range': None,
        'temperature': None,
        'thermal_reasoning': None,
        'thermal_confidence': None,
    }

    start = time.time()
    output=HostMetadata(state)
    end=time.time()
    duration=(end-start)/60
    print(f'Accession: {accession}\nHost Species: {output.get("host_species", "unknown")}')
    print(f'Duration (minutes): {round(duration,2)}')
    print(f'LLM Output: {output}\n\n')

    with open(f'results/host/host_classification.csv', 'a') as csvfile1:
        csvfile1.write(f"{accession},{output.get('host_species','unknown')},{output.get('taxon_level','unknown')},{output.get('host_reasoning','unknown')},{round(duration,2)}\n")

    with open(f'results/host/host_classification.txt', 'a') as f:
        f.write(f'==============={accession}==============\n')
        f.write(f'Host Species: {output.get("host_species", "unknown")}\n')
        f.write(f'Taxonomic Level: {output.get("taxon_level", "unknown")}\n')
        f.write(f'Reasoning: {output.get("host_reasoning", "unknown")}\n')
        f.write(f'Duration: {round(duration,2)} minutes\n')


full_end=time.time()
full_duration=(full_end-full_Start)/60

with open('results/host/host_classification.txt', 'a') as outf:
    outf.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}\n')


==============ADI87650=============
Metadata: {'uid': '298571302', 'accession': 'ADI87650', 'title': 'major head protein, partial [Tequatrovirus T4]', 'organism': 'Tequatrovirus T4', 'taxonId': 10665, 'sequence_length': 141, 'create_date': '2010/06/23', 'update_date': '2016/07/25', 'dbsource': 'insd', 'extra': 'gi|298571302|gb|ADI87650.1|', 'host': 'Enterobacteriales', 'old_name': 'Escherichia virus T4', 'map': 'between gp22 and SegD', 'clone': 's10-10', 'country': 'China: Donghu Lake, Wuhan'}
content='' additional_kwargs={} response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-27T22:33:29.126852Z', 'done': True, 'done_reason': 'length', 'total_duration': 156622260833, 'load_duration': 119811958, 'prompt_eval_count': 550, 'prompt_eval_duration': 1618798625, 'eval_count': 4096, 'eval_duration': 153361044855, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'} id='lc_run--019dd111-1b56-7ef3-ad16-2b840fc5ba69-0' tool_calls=[] invalid_tool_calls=[] usage_metadata

In [13]:
 # Identify thermal range from metadata
with open('results/thermal/thermal_metadata_classification.txt', 'w') as outf:
     outf.write(f'Thermal classification of example data using accession metadata\n{datetime.now()}\n\n')

with open('results/thermal/thermal_classification.csv', 'w') as csvout:
    csvout.write('accession,thermal_range,inference_type,reasoning,confidence,duration(m)\n')

full_Start=time.time()
for accession in accessions:
    print(f'=============={accession}=============')
    with open(f"data/library/{accession}/{accession}.json", "r") as f:
        metadata = json.load(f)
        print(f'Metadata: {metadata}')

        # Define the start
    state = {
        'paper_text': None,
        'source_path': None,
        'model': 'qwen3.5',
        'metadata': metadata, # only including metadata
        'host_species': None,
        'reasoning': None,
        'confidence': None,
        'thermal_range': None,
        'temperature': None,
        'thermal_reasoning': None,
        'thermal_confidence': None,
    }

    start = time.time()
    output=ThermalMetadata(state)
    end=time.time()
    duration=(end-start)/60
    print(f'\nThermal Range: {output.get("thermal_range", "unknown")}')
    print(f'Duration (minutes): {round(duration,2)}')
    print(f'LLM Output: {output}\n\n')

    reasoning=output.get("reasoning", "unknown")
    reasoning=reasoning.replace(',', '')

    with open(f'results/thermal/thermal_metadata_classification.txt', 'a') as f:
        f.write(f'==============={accession}==============\n')
        f.write(f'Thermal Range {output.get("thermal_range", "unknown")}\n')
        f.write(f'Inference Type {output.get("inference_type", "unknown")}\n')
        f.write(f'Reasoning {output.get("reasoning", "unknown")}\n')
        f.write(f'Confidence {output.get("confidence", "unknown")}\n')

    with open(f'results/thermal/thermal_classification.csv', 'a') as f:
        f.write(f"{accession},{output.get('thermal_range', 'unknown')},{output.get('inference_type', 'unknown')},{reasoning},{output.get('confidence', 'unknown')},{round(duration,2)}\n")

full_end=time.time()
full_duration=(full_end-full_Start)/60

with open('results/thermal/thermal_metadata_classification.txt', 'a') as f:
    f.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}\n')

==============ADI87650=============
Metadata: {'uid': '298571302', 'accession': 'ADI87650', 'title': 'major head protein, partial [Tequatrovirus T4]', 'organism': 'Tequatrovirus T4', 'taxonId': 10665, 'sequence_length': 141, 'create_date': '2010/06/23', 'update_date': '2016/07/25', 'dbsource': 'insd', 'extra': 'gi|298571302|gb|ADI87650.1|', 'host': 'Enterobacteriales', 'old_name': 'Escherichia virus T4', 'map': 'between gp22 and SegD', 'clone': 's10-10', 'country': 'China: Donghu Lake, Wuhan'}
content='' additional_kwargs={} response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-27T23:09:27.989651Z', 'done': True, 'done_reason': 'length', 'total_duration': 156878840667, 'load_duration': 101063833, 'prompt_eval_count': 601, 'prompt_eval_duration': 1689082459, 'eval_count': 4096, 'eval_duration': 153572354781, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'} id='lc_run--019dd132-0b61-7a63-931f-141f28bd6843-0' tool_calls=[] invalid_tool_calls=[] usage_metadata